In [1]:
# ─────────────────────────────────────────────
# PHASE 2 — Data Loading
# ─────────────────────────────────────────────

import pandas as pd
import numpy as np
import os

# Load both sheets from the Excel file
# The dataset has 2 sheets — one per year range
# We'll combine them into one big DataFrame

file_path = "../data/raw/online_retail_II.xlsx"

print("Loading Sheet 1 (2009–2010)... this may take a minute ☕")
df1 = pd.read_excel(file_path, sheet_name="Year 2009-2010")

print("Loading Sheet 2 (2010–2011)...")
df2 = pd.read_excel(file_path, sheet_name="Year 2010-2011")

# Stack both sheets on top of each other (same columns, more rows)
df = pd.concat([df1, df2], ignore_index=True)

print(f"\n✅ Data loaded successfully!")
print(f"Total rows: {df.shape[0]:,}")
print(f"Total columns: {df.shape[1]}")

Loading Sheet 1 (2009–2010)... this may take a minute ☕
Loading Sheet 2 (2010–2011)...

✅ Data loaded successfully!
Total rows: 1,067,371
Total columns: 8


In [2]:
# ─────────────────────────────────────────────
# STEP 2 — First Look at the Data
# ─────────────────────────────────────────────


print("=== First 5 Rows ===")
display(df.head())

# See all column names, their data types, and non-null counts
# This is your first clue about missing values and wrong types
print("\n=== Column Info ===")
df.info()

# Basic statistics for all numeric columns
# Min, max, mean, std — great for spotting weird values
print("\n=== Basic Statistics ===")
display(df.describe())

=== First 5 Rows ===


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom



=== Column Info ===
<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 65.1+ MB

=== Basic Statistics ===


,Quantity,InvoiceDate,Price,Customer ID
count,1.067371e+06,1067371,1.067371e+06,824364.000000
mean,9.938898e+00,2011-01-02 21:13:55.394029,4.649388e+00,15324.638504
min,-8.099500e+04,2009-12-01 07:45:00,-5.359436e+04,12346.000000
25%,1.000000e+00,2010-07-09 09:46:00,1.250000e+00,13975.000000
50%,3.000000e+00,2010-12-07 15:28:00,2.100000e+00,15255.000000
75%,1.000000e+01,2011-07-22 10:23:00,4.150000e+00,16797.000000
max,8.099500e+04,2011-12-09 12:50:00,3.897000e+04,18287.000000
std,1.727058e+02,NaN,1.235531e+02,1697.464450


In [3]:
# ─────────────────────────────────────────────
# FIX 1 — Remove Cancelled Orders & Bad Values
# ─────────────────────────────────────────────

# Before we clean, always record the original shape
# So we can see exactly how many rows each step removes
rows_before = len(df)
print(f"Rows before cleaning: {rows_before:,}")

# --- Remove Cancelled Orders ---
# Invoices starting with "C" are cancellations
# The ~ symbol means "NOT" — keep rows where Invoice does NOT start with "C"
df = df[~df["Invoice"].astype(str).str.startswith("C")]
print(f"Rows after removing cancellations: {len(df):,} (removed {rows_before - len(df):,} rows)")

# --- Remove rows where Quantity is 0 or negative ---
# A sale must have at least 1 unit sold
# Negative quantities = returns/errors, zero = nothing sold
rows_before = len(df)
df = df[df["Quantity"] > 0]
print(f"Rows after removing Quantity ≤ 0: {len(df):,} (removed {rows_before - len(df):,} rows)")

# --- Remove rows where Price is 0 or negative ---
# Every real product must have a positive price
# Zero price = free samples, test entries, or errors
rows_before = len(df)
df = df[df["Price"] > 0]
print(f"Rows after removing Price ≤ 0: {len(df):,} (removed {rows_before - len(df):,} rows)")

print(f"\n✅ Fix 1 Complete!")
print(f"Final rows after Fix 1: {len(df):,}")

Rows before cleaning: 1,067,371
Rows after removing cancellations: 1,047,877 (removed 19,494 rows)
Rows after removing Quantity ≤ 0: 1,044,420 (removed 3,457 rows)
Rows after removing Price ≤ 0: 1,041,670 (removed 2,750 rows)

✅ Fix 1 Complete!
Final rows after Fix 1: 1,041,670


In [4]:
# ─────────────────────────────────────────────
# FIX 2 — Fix Customer ID Data Type
# ─────────────────────────────────────────────

# Customer ID currently looks like 17850.0 (float)
# It should look like "17850" (string)
# Why string? Because we never do math on IDs
# You'd never calculate average CustomerID — it means nothing

# Step 1 — See what it looks like before
print("Before fix:")
print(df["Customer ID"].head(10))
print(f"Data type: {df['Customer ID'].dtype}")

# Step 2 — Convert float to string
# First convert to Int64 (removes decimal) then to string
# We use "Int64" with capital I — this version supports NaN values
# Regular "int64" would crash if there are any nulls
df["Customer ID"] = df["Customer ID"].astype("Int64").astype("str")

# Step 3 — Replace the string "nan" with actual NaN
# When we converted nulls to string, they became the text "nan"
# We need to convert them back to real null values
df["Customer ID"] = df["Customer ID"].replace("nan", pd.NA)

# Step 4 — See what it looks like after
print("\nAfter fix:")
print(df["Customer ID"].head(10))
print(f"Data type: {df['Customer ID'].dtype}")

Before fix:
0    13085.0
1    13085.0
2    13085.0
3    13085.0
4    13085.0
5    13085.0
6    13085.0
7    13085.0
8    13085.0
9    13085.0
Name: Customer ID, dtype: float64
Data type: float64

After fix:
0    13085
1    13085
2    13085
3    13085
4    13085
5    13085
6    13085
7    13085
8    13085
9    13085
Name: Customer ID, dtype: str
Data type: str


In [5]:
# ─────────────────────────────────────────────
# FIX 3 — Fill Missing Descriptions via StockCode
# ─────────────────────────────────────────────

# How many descriptions are missing right now?
missing_before = df["Description"].isna().sum()
print(f"Missing descriptions before fix: {missing_before:,}")

# For each StockCode, find the most commonly used Description
# This is the "group mode" technique
# .groupby("StockCode") → group all rows by StockCode
# ["Description"] → look at the Description column within each group
# .transform(lambda x: x.mode()[0] if not x.mode().empty else pd.NA)
# → for each group, find the most frequent description (mode)
# → if no description exists at all for that StockCode, leave it as NA

df["Description"] = df.groupby("StockCode")["Description"].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else pd.NA)
)

# How many are still missing after fix?
missing_after = df["Description"].isna().sum()
print(f"Missing descriptions after fix: {missing_after:,}")
print(f"Descriptions filled: {missing_before - missing_after:,}")

print("\n✅ Fix 3 Complete!")

Missing descriptions before fix: 0
Missing descriptions after fix: 0
Descriptions filled: 0

✅ Fix 3 Complete!


In [6]:
# ─────────────────────────────────────────────
# FIX 4 — Remove Duplicate Rows
# ─────────────────────────────────────────────

# How many duplicate rows exist?
# A duplicate = every column has identical values to another row
duplicates = df.duplicated().sum()
print(f"Duplicate rows found: {duplicates:,}")

# Remove duplicates — keep the first occurrence, drop the rest
rows_before = len(df)
df = df.drop_duplicates()
print(f"Rows before: {rows_before:,}")
print(f"Rows after: {len(df):,}")
print(f"Rows removed: {rows_before - len(df):,}")

print("\n✅ Fix 4 Complete!")

Duplicate rows found: 33,757
Rows before: 1,041,670
Rows after: 1,007,913
Rows removed: 33,757

✅ Fix 4 Complete!


In [7]:
df.isnull().sum()

Invoice             0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
Price               0
Customer ID    228488
Country             0
dtype: int64

In [8]:
# ─────────────────────────────────────────────
# FIX 5 — Final Health Check
# ─────────────────────────────────────────────

print("=== Final DataFrame Shape ===")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

print("\n=== Missing Values Per Column ===")
print(df.isnull().sum())

print("\n=== Data Types ===")
print(df.dtypes)

print("\n=== Sample of Clean Data ===")
display(df.head())

print("\n=== Basic Statistics ===")
display(df.describe())

# Verify no negative quantities or prices snuck through
print("\n=== Sanity Checks ===")
print(f"Rows with Quantity ≤ 0: {(df['Quantity'] <= 0).sum()}")
print(f"Rows with Price ≤ 0: {(df['Price'] <= 0).sum()}")
print(f"Cancelled invoices remaining: {df['Invoice'].astype(str).str.startswith('C').sum()}")

=== Final DataFrame Shape ===
Rows: 1,007,913
Columns: 8

=== Missing Values Per Column ===
Invoice             0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
Price               0
Customer ID    228488
Country             0
dtype: int64

=== Data Types ===
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID               str
Country                   str
dtype: object

=== Sample of Clean Data ===


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom



=== Basic Statistics ===


,Quantity,InvoiceDate,Price
count,1.007913e+06,1007913,1.007913e+06
mean,1.111718e+01,2011-01-04 10:34:18.759595,4.074252e+00
min,1.000000e+00,2009-12-01 07:45:00,1.000000e-03
25%,1.000000e+00,2010-07-06 11:21:00,1.250000e+00
50%,4.000000e+00,2010-12-09 15:29:00,2.100000e+00
75%,1.200000e+01,2011-07-28 12:29:00,4.130000e+00
max,8.099500e+04,2011-12-09 12:50:00,2.511109e+04
std,1.284700e+02,NaN,5.043045e+01



=== Sanity Checks ===
Rows with Quantity ≤ 0: 0
Rows with Price ≤ 0: 0
Cancelled invoices remaining: 0


In [9]:
# ─────────────────────────────────────────────
# SAVE — Export Cleaned Data
# ─────────────────────────────────────────────

import os

# Create the processed folder if it doesn't exist
os.makedirs("../data/processed", exist_ok=True)

# Save as CSV — much faster to load than Excel
# index=False means don't save the row numbers as a column
df.to_csv("../data/processed/online_retail_cleaned.csv", index=False)

print("✅ Cleaned data saved to data/processed/online_retail_cleaned.csv")
print(f"Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

✅ Cleaned data saved to data/processed/online_retail_cleaned.csv
Final shape: 1,007,913 rows × 8 columns


In [10]:
# ─────────────────────────────────────────────
# STEP 4 — Feature Engineering
# ─────────────────────────────────────────────

# --- Time-based features ---
# Extract useful parts from InvoiceDate
# These help the model learn seasonal and time patterns

df["Year"] = df["InvoiceDate"].dt.year
# dt.year extracts just the year number → 2009, 2010, 2011

df["Month"] = df["InvoiceDate"].dt.month
# dt.month extracts month number → 1 to 12

df["Day"] = df["InvoiceDate"].dt.day
# dt.day extracts day of month → 1 to 31

df["DayOfWeek"] = df["InvoiceDate"].dt.dayofweek
# dt.dayofweek → Monday=0, Tuesday=1, ... Sunday=6

df["Hour"] = df["InvoiceDate"].dt.hour
# dt.hour extracts hour → 0 to 23

df["WeekOfYear"] = df["InvoiceDate"].dt.isocalendar().week.astype(int)
# ISO week number → 1 to 52
# Useful for spotting patterns like "week 48 is always busy"

# --- Business metric features ---
# These are calculated columns that represent real business value

df["TotalPrice"] = df["Quantity"] * df["Price"]
# Revenue per line item
# If someone bought 6 mugs at £2.50 each → TotalPrice = £15.00
# This is the most important metric for sales analysis

# --- Quick verification ---
print("=== New Columns Added ===")
print(df[["InvoiceDate", "Year", "Month", "Day", 
          "DayOfWeek", "Hour", "WeekOfYear", "TotalPrice"]].head(10))

print(f"\nTotal columns now: {df.shape[1]}")
print("\n✅ Feature Engineering Complete!")

=== New Columns Added ===
          InvoiceDate  Year  Month  Day  DayOfWeek  Hour  WeekOfYear  \
0 2009-12-01 07:45:00  2009     12    1          1     7          49   
1 2009-12-01 07:45:00  2009     12    1          1     7          49   
2 2009-12-01 07:45:00  2009     12    1          1     7          49   
3 2009-12-01 07:45:00  2009     12    1          1     7          49   
4 2009-12-01 07:45:00  2009     12    1          1     7          49   
5 2009-12-01 07:45:00  2009     12    1          1     7          49   
6 2009-12-01 07:45:00  2009     12    1          1     7          49   
7 2009-12-01 07:45:00  2009     12    1          1     7          49   
8 2009-12-01 07:46:00  2009     12    1          1     7          49   
9 2009-12-01 07:46:00  2009     12    1          1     7          49   

   TotalPrice  
0        83.4  
1        81.0  
2        81.0  
3       100.8  
4        30.0  
5        39.6  
6        30.0  
7        59.5  
8        30.6  
9        45.0  

Tota

In [11]:
# ─────────────────────────────────────────────
# SAVE — Final Version with Features
# ─────────────────────────────────────────────

df.to_csv("../data/processed/online_retail_cleaned.csv", index=False)

print("✅ Final dataset saved!")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("\nColumns saved:")
for col in df.columns:
    print(f"  → {col}")

✅ Final dataset saved!
Shape: 1,007,913 rows × 15 columns

Columns saved:
  → Invoice
  → StockCode
  → Description
  → Quantity
  → InvoiceDate
  → Price
  → Customer ID
  → Country
  → Year
  → Month
  → Day
  → DayOfWeek
  → Hour
  → WeekOfYear
  → TotalPrice
